# Unit Economics of an AI Agent

How do you work out the **per-request AWS cost** of an AI agent workload?

This notebook walks through a PDF-translation agent one service at a time, then
adds them up. Every rate lives in [`cost.py`](cost.py) — edit it there for your region.

The request flows through:

1. **Textract** — read text blocks from the PDF (priced per page)
2. **Bedrock (Nova Lite)** — translate the blocks (priced per token)
3. **Lambda** — the compute running each step (priced per GB-second)
4. **S3** — storing/reading files (priced per request)
5. **AgentCore Gateway + Runtime** — the agent host (estimates)

In [ ]:
import cost

# The rates we're working with (USD)
print(f"Textract, per page:       ${cost.TEXTRACT_COST_PER_PAGE}")
print(f"Bedrock in,  per 1K tok:  ${cost.NOVA_LITE_INPUT_COST_PER_1K}")
print(f"Bedrock out, per 1K tok:  ${cost.NOVA_LITE_OUTPUT_COST_PER_1K}")
print(f"Lambda, per GB-second:    ${cost.LAMBDA_COST_PER_GB_SECOND}")
print(f"S3 GET / PUT:             ${cost.S3_GET_COST} / ${cost.S3_PUT_COST}")

## Describe one request

A `Request` bundles everything about a single request that drives its cost:
how many pages, how many tokens, how long Lambda ran, and so on.

In [ ]:
req = cost.Request(
    pages=3,
    input_tokens=4000,
    output_tokens=4000,
    lambda_memory_mb=1024,
    lambda_duration_ms=3000,
)
req

## 1. Textract — priced per page

The simplest component: `pages × rate`.

In [ ]:
textract = req.pages * cost.TEXTRACT_COST_PER_PAGE
print(f"{req.pages} pages × ${cost.TEXTRACT_COST_PER_PAGE} = ${textract:.6f}")

## 2. Bedrock — priced per token

Input and output tokens are billed at **different** rates (output is pricier).
Note the `/ 1000` — rates are quoted per 1,000 tokens.

In [ ]:
bedrock = (
    req.input_tokens / 1000 * cost.NOVA_LITE_INPUT_COST_PER_1K
    + req.output_tokens / 1000 * cost.NOVA_LITE_OUTPUT_COST_PER_1K
)
print(f"in:  {req.input_tokens} tok → ${req.input_tokens / 1000 * cost.NOVA_LITE_INPUT_COST_PER_1K:.6f}")
print(f"out: {req.output_tokens} tok → ${req.output_tokens / 1000 * cost.NOVA_LITE_OUTPUT_COST_PER_1K:.6f}")
print(f"bedrock total = ${bedrock:.6f}")

## 3. Lambda — priced per GB-second

This is the one that trips people up. Lambda bills **GB-seconds**: the memory you
allocate (in GB) times how long the function ran (in seconds).

$$\text{GB-seconds} = \frac{\text{memory MB}}{1024} \times \frac{\text{duration ms}}{1000}$$

In [ ]:
gb_seconds = (req.lambda_memory_mb / 1024) * (req.lambda_duration_ms / 1000)
lambda_cost = gb_seconds * cost.LAMBDA_COST_PER_GB_SECOND
print(f"{req.lambda_memory_mb} MB for {req.lambda_duration_ms} ms = {gb_seconds:.4f} GB-s")
print(f"{gb_seconds:.4f} GB-s × ${cost.LAMBDA_COST_PER_GB_SECOND} = ${lambda_cost:.8f}")

## 4. S3, Gateway, Runtime — per request

The remaining pieces are flat per-request or per-invocation charges.

In [ ]:
s3 = req.s3_get_requests * cost.S3_GET_COST + req.s3_put_requests * cost.S3_PUT_COST
gateway = req.gateway_invocations * cost.GATEWAY_COST_PER_INVOCATION
runtime = cost.RUNTIME_COST_ESTIMATE
print(f"S3:      ${s3:.8f}")
print(f"Gateway: ${gateway:.6f}")
print(f"Runtime: ${runtime:.6f}")

## Add it all up

`calculate_cost()` does exactly the arithmetic above and returns an itemised dict.

In [ ]:
costs = cost.calculate_cost(req)
print(cost.format_report(costs))

## Scale it: what does this cost at volume?

One request is fractions of a cent — the interesting question is the monthly bill.

In [ ]:
per_request = costs["totalEstimated"]
for n in (1_000, 100_000, 1_000_000):
    print(f"{n:>10,} requests/mo  →  ${per_request * n:,.2f}")

## Where does the money go?

Breaking the total into shares shows which service to optimise first.

In [ ]:
total = costs["totalEstimated"]
components = ["textract", "bedrock", "lambda", "s3", "agentcoreGateway", "agentcoreRuntime"]
for name in components:
    part = costs[name]["total"]
    bar = "█" * int(round(part / total * 40))
    print(f"{name:<18} {part/total*100:5.1f}%  {bar}")

## Try your own numbers

Change the inputs below and re-run to see the effect.

In [ ]:
my_req = cost.Request(pages=10, input_tokens=12_000, output_tokens=9_000)
print(cost.format_report(cost.calculate_cost(my_req)))